# Path B — Direct Redshift Pull (`psycopg2`)

> This is the **alternative** data-access path. The main module path is **Path A** in `eda_example.ipynb`, which uses `data_io.load_data()` (boto3 `redshift-data` + `UNLOAD` to S3, IAM auth, no VPN). See its Setup section for the comparison.

**Use this notebook when**: you need a quick one-off sample on your laptop and you already have VPN + DB credentials. The output Parquet is what gets committed to `sample_data_from_redshift/` for offline development.

## Steps

1. **Connect to VPN** — `vpn.ao.zapsi.net`.
2. **Verify the connection** in the AWS Redshift Query Editor:
   <https://af-south-1.console.aws.amazon.com/sqlworkbench/home?region=af-south-1#/client>
3. **Provide the password via environment variable** — never hardcode it in the notebook.
   Create a `.env` file in this folder (already in `.gitignore`):
   ```
   REDSHIFT_PASSWORD=...
   ```
   Or export it in your shell before launching Jupyter:
   ```bash
   export REDSHIFT_PASSWORD='...'
   ```
4. **Install dependencies** (if not already installed via `requirements.txt`):
   ```bash
   pip install python-dotenv psycopg2-binary pyarrow
   ```
   Restart the kernel after installing.
5. **Run the cell below** — it connects, runs the sample query, and saves the result to `../sample_data_from_redshift/`.

⚠️ **Security**: the password is a real production credential. Read it from the environment, never paste it into the notebook. If it ever lands in a committed cell, **rotate it immediately** — git history retains it forever.

In [3]:
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd

# Load environment variables from .env file
load_dotenv()

# ====== CONFIG ======
REDSHIFT_HOST = "redshift-cluster-dsi.cl4o4mmtx9ir.af-south-1.redshift.amazonaws.com"
REDSHIFT_PORT = 5439
REDSHIFT_DB   = "prod"
REDSHIFT_USER = "awsuser"

# put your password in an env var before running:
# export REDSHIFT_PASSWORD='your-password-here'
#REDSHIFT_PASSWORD = os.getenv('REDSHIFT_PASSWORD')
REDSHIFT_PASSWORD = "RFMPXefkbh465%."

# Example query – change this to your table
QUERY = "SELECT * FROM prod.dth_churn_ml_training.training_features WHERE RANDOM() < 0.003;"

OUTPUT_PARQUET = "../sample_data_from_redshift/sample_from_prod.parquet"
# ====================


if not REDSHIFT_PASSWORD:
    raise RuntimeError("REDSHIFT_PASSWORD env var not set")

conn = psycopg2.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    dbname=REDSHIFT_DB,
    user=REDSHIFT_USER,
    password=REDSHIFT_PASSWORD,
)

try:
    # read into pandas
    df = pd.read_sql(QUERY, conn)

    print(f"Fetched {len(df)} rows from Redshift")
    print(df.head())

    # save locally
    df.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"Saved {len(df)} rows to {OUTPUT_PARQUET}")
finally:
    conn.close()

/var/folders/3d/dh5fxyvd55sbf5r8pv3bfclw0000gq/T/ipykernel_67316/3993704429.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(QUERY, conn)


Fetched 110069 rows from Redshift
   idconsumo  id_contaservico codigocontaservico  idconta iddim_date_inicio  \
0  487608880          4172747       100002790201  4128367        2025-02-24   
1  597066760              441       100004310101      431        2026-02-24   
2  502340486            53241       100006150201    50135        2025-04-17   
3  490204594          4301417       100014220501  4259229        2025-03-04   
4  588341743          1892316       100026070201  1840684        2026-01-26   

  iddim_date_fim  id_produto_actual tipo_produto_actual  tipo_subscricao  \
0     2025-03-02                 24            tafacil7                7   
1     2026-03-05                 24             tafacil                7   
2     2025-04-30                 23             tafacil                7   
3     2025-03-10                 22            tafacil7                7   
4     2026-02-01                 24            tafacil7                7   

  tipo_stb  ...  was_contacted  to